In [13]:
import os

import numpy as np
import open3d as o3d

# Define the filename of your exported STL file
stl_filename = "/Users/nils/Desktop/assemblies/tc_assy1.stl"  # Make sure this matches your filename

# --- Load the STL file into Open3D ---
mesh = o3d.io.read_triangle_mesh(stl_filename)
if mesh.is_empty():
    print(f"Error: Could not load mesh from {stl_filename}")
else:
    print(f"Successfully loaded mesh from {stl_filename}")
    bbox = mesh.get_axis_aligned_bounding_box()
    min_bound = bbox.get_min_bound()
    max_bound = bbox.get_max_bound()
    center = bbox.get_center()

Successfully loaded mesh from /Users/nils/Desktop/assemblies/tc_assy1.stl


In [14]:
# --- Create the Flat Point Cloud Surface at Chassis Height ---
# Estimate the chassis height.
chassis_z_height = min_bound[2] + (max_bound[2] - min_bound[2]) * 0.15  # 5% above min Z

# Define the extent of the flat surface based on the car's width and length
flat_cloud_x_min, flat_cloud_x_max = (
    min_bound[0] - (max_bound[0] - min_bound[0]) * 0.1,
    max_bound[0] + (max_bound[0] - min_bound[0]) * 0.1,
)  # Extend slightly
flat_cloud_y_min, flat_cloud_y_max = (
    min_bound[1] - (max_bound[1] - min_bound[1]) * 0.1,
    max_bound[1] + (max_bound[1] - min_bound[1]) * 0.1,
)  # Extend slightly

# Generate a grid of points for the flat surface
n_x_points = 100  # Number of points in X direction
n_y_points = 150  # Number of points in Y direction
x_vals = np.linspace(flat_cloud_x_min, flat_cloud_x_max, n_x_points)
y_vals = np.linspace(flat_cloud_y_min, flat_cloud_y_max, n_y_points)
xx, yy = np.meshgrid(x_vals, y_vals)

flat_points = np.zeros((n_x_points * n_y_points, 3))
flat_points[:, 0] = xx.flatten()
flat_points[:, 1] = yy.flatten()
flat_points[:, 2] = chassis_z_height  # All points at the chassis height

flat_point_cloud = o3d.geometry.PointCloud()
flat_point_cloud.points = o3d.utility.Vector3dVector(flat_points)
flat_point_cloud.paint_uniform_color([0, 0, 1])  # Paint flat cloud blue

print(
    f"Generated flat point cloud surface with {len(flat_point_cloud.points)} points at Z = {chassis_z_height:.2f}"
)

# --- Create the Half-Ellipsoid Point Cloud Shell ---
# Define the center of the half-ellipsoid.
# We'll place its Z-center at the chassis height for a potentially connected look.
half_ellipsoid_center = np.array([center[0], center[1], chassis_z_height])

# Define the radii of the ellipsoid along the X, Y, and Z axes
# Adjust these values to control the shape and size of the half-ellipsoid shell
a = (max_bound[0] - min_bound[0]) * 0.55  # X-radius (55% of car's X extent)
b = (max_bound[1] - min_bound[1]) * 0.6  # Y-radius (55% of car's Y extent)
# Z-radius extends from the chassis height to cover the top of the car
c = (max_bound[2] - chassis_z_height) + (
    max_bound[2] - min_bound[2]
) * 0.1  # Radius to top + 10% of Z extent padding

# Generate points on the surface of the upper half of an ellipsoid
n_theta = 100  # Number of samples around the circumference (0 to 2*pi)
n_phi = 50  # Number of samples along the height (0 to pi/2 for upper half)

theta_vals = np.linspace(0, 2 * np.pi, n_theta)
phi_vals = np.linspace(0, np.pi / 2, n_phi)  # Only upper half

theta, phi = np.meshgrid(theta_vals, phi_vals)

# Parametric equations for ellipsoid surface centered at (cx, cy, cz) with radii (a, b, c)
# X = cx + a * sin(phi) * cos(theta)
# Y = cy + b * sin(phi) * sin(theta)
# Z = cz + c * cos(phi)

half_ellipsoid_points = np.zeros((n_theta * n_phi, 3))
half_ellipsoid_points[:, 0] = (
    half_ellipsoid_center[0] + a * np.sin(phi).flatten() * np.cos(theta).flatten()
)
half_ellipsoid_points[:, 1] = (
    half_ellipsoid_center[1] + b * np.sin(phi).flatten() * np.sin(theta).flatten()
)
half_ellipsoid_points[:, 2] = half_ellipsoid_center[2] + c * np.cos(phi).flatten()


half_ellipsoid_point_cloud = o3d.geometry.PointCloud()
half_ellipsoid_point_cloud.points = o3d.utility.Vector3dVector(half_ellipsoid_points)
half_ellipsoid_point_cloud.paint_uniform_color([1, 0, 0])  # Paint red

print(
    f"Generated half-ellipsoid point cloud shell with {len(half_ellipsoid_point_cloud.points)} points"
)
print(
    f"Half-ellipsoid centered at {half_ellipsoid_center.round(2)} with radii ({a:.2f}, {b:.2f}, {c:.2f})"
)


# --- Combine the Point Clouds ---
combined_point_cloud = o3d.geometry.PointCloud()
combined_point_cloud.points.extend(flat_point_cloud.points)
combined_point_cloud.points.extend(half_ellipsoid_point_cloud.points)
combined_point_cloud.colors.extend(flat_point_cloud.colors)
combined_point_cloud.colors.extend(half_ellipsoid_point_cloud.colors)


# --- Visualize the results ---
print("Visualizing mesh and generated point clouds (press 'q' to close the window)")
o3d.visualization.draw_geometries([mesh, combined_point_cloud])

# Optional: Visualize only the combined point cloud
# print("Visualizing combined point cloud (press 'q' to close the window)")
# o3d.visualization.draw_geometries([combined_point_cloud])

# You can now work with the 'combined_point_cloud' object in Open3D
# For example, you can access the points as a NumPy array:
# all_points = np.asarray(combined_point_cloud.points)
# all_colors = np.asarray(combined_point_cloud.colors)

Generated flat point cloud surface with 15000 points at Z = 146.22
Generated half-ellipsoid point cloud shell with 5000 points
Half-ellipsoid centered at [ 91.5    0.   146.22] with radii (1307.35, 664.80, 925.61)
Visualizing mesh and generated point clouds (press 'q' to close the window)


In [6]:
# --- Create the Flat Point Cloud at Chassis Height ---
# Estimate the chassis height. This is a simplification;
# you might need to adjust the offset based on your specific car model.
# We'll place it slightly above the minimum Z bound.
chassis_z_height = min_bound[2] + (max_bound[2] - min_bound[2]) * 0.15  # 5% above min Z

# Define the extent of the flat cloud based on the car's width and length
flat_cloud_x_min, flat_cloud_x_max = (
    min_bound[0] - (max_bound[0] - min_bound[0]) * 0.1,
    max_bound[0] + (max_bound[0] - min_bound[0]) * 0.1,
)  # Extend slightly
flat_cloud_y_min, flat_cloud_y_max = (
    min_bound[1] - (max_bound[1] - min_bound[1]) * 0.1,
    max_bound[1] + (max_bound[1] - min_bound[1]) * 0.1,
)  # Extend slightly

# Number of points for the flat cloud
n_flat_points = 1000

# Generate random points within the defined X-Y range at the chassis height
flat_points = np.zeros((n_flat_points, 3))
flat_points[:, 0] = np.random.uniform(flat_cloud_x_min, flat_cloud_x_max, n_flat_points)
flat_points[:, 1] = np.random.uniform(flat_cloud_y_min, flat_cloud_y_max, n_flat_points)
flat_points[:, 2] = chassis_z_height  # All points at the chassis height

flat_point_cloud = o3d.geometry.PointCloud()
flat_point_cloud.points = o3d.utility.Vector3dVector(flat_points)
flat_point_cloud.paint_uniform_color([0, 0, 1])  # Paint flat cloud blue
print(
    f"Generated flat point cloud with {len(flat_point_cloud.points)} points at Z = {chassis_z_height:.2f}"
)

Generated flat point cloud with 1000 points at Z = 146.22


In [7]:
# --- Create the Half-Ellipsoid Point Cloud for the Cover ---
# Define the center of the half-ellipsoid. We'll place it above the car's center.
half_ellipsoid_center = np.array(
    [center[0], center[1], min_bound[2] + (max_bound[2] - min_bound[2]) * 0.6]
)  # 60% up from min Z

# Define the radii of the ellipsoid along the X, Y, and Z axes
# Adjust these values to control the shape and size of the half-ellipsoid
a = (max_bound[0] - min_bound[0]) * 0.6  # X-radius (60% of car's X extent)
b = (max_bound[1] - min_bound[1]) * 0.6  # Y-radius (60% of car's Y extent)
c = (max_bound[2] - min_bound[2]) * 0.5  # Z-radius (50% of car's total Z extent)

# Number of points for the half-ellipsoid cloud
n_half_ellipsoid_points = 1500  # You can adjust this

# Define the bounding box for generating random points that covers the half-ellipsoid
# The half-ellipsoid is defined by (x/a)^2 + (y/b)^2 + (z/c)^2 <= 1 with z >= center_z
gen_bbox_min = np.array(
    [
        half_ellipsoid_center[0] - a,
        half_ellipsoid_center[1] - b,
        half_ellipsoid_center[2],
    ]
)
gen_bbox_max = np.array(
    [
        half_ellipsoid_center[0] + a,
        half_ellipsoid_center[1] + b,
        half_ellipsoid_center[2] + c,
    ]
)

# Generate random points within this bounding box
random_points_in_bbox = np.random.uniform(
    low=gen_bbox_min, high=gen_bbox_max, size=(n_half_ellipsoid_points * 2, 3)
)  # Generate more and filter

# Filter points to keep only those inside the half-ellipsoid volume
# The ellipsoid equation: ((x-cx)/a)^2 + ((y-cy)/b)^2 + ((z-cz)/c)^2 <= 1
# And the half-ellipsoid condition: z >= cz
ellipsoid_condition = (
    ((random_points_in_bbox[:, 0] - half_ellipsoid_center[0]) / a) ** 2
    + ((random_points_in_bbox[:, 1] - half_ellipsoid_center[1]) / b) ** 2
    + ((random_points_in_bbox[:, 2] - half_ellipsoid_center[2]) / c) ** 2
) <= 1

half_ellipsoid_condition = random_points_in_bbox[:, 2] >= half_ellipsoid_center[2]

points_in_half_ellipsoid = random_points_in_bbox[
    ellipsoid_condition & half_ellipsoid_condition
]

# If we didn't get enough points inside the half-ellipsoid, generate more or just use what we have
if len(points_in_half_ellipsoid) < n_half_ellipsoid_points:
    print(
        f"Warning: Generated only {len(points_in_half_ellipsoid)} points inside the half-ellipsoid. Consider increasing initial random points."
    )
    # Use the points generated, or regenerate with a higher multiplier if needed
    final_half_ellipsoid_points = points_in_half_ellipsoid
else:
    # Take the desired number of points if we generated more than needed
    final_half_ellipsoid_points = points_in_half_ellipsoid[:n_half_ellipsoid_points]


half_ellipsoid_point_cloud = o3d.geometry.PointCloud()
half_ellipsoid_point_cloud.points = o3d.utility.Vector3dVector(
    final_half_ellipsoid_points
)
half_ellipsoid_point_cloud.paint_uniform_color([1, 0, 0])  # Paint red

print(
    f"Generated half-ellipsoid point cloud with {len(half_ellipsoid_point_cloud.points)} points"
)
print(
    f"Half-ellipsoid centered at {half_ellipsoid_center.round(2)} with radii ({a:.2f}, {b:.2f}, {c:.2f})"
)


# --- Combine the Point Clouds ---
combined_point_cloud = o3d.geometry.PointCloud()
combined_point_cloud.points.extend(flat_point_cloud.points)
combined_point_cloud.points.extend(half_ellipsoid_point_cloud.points)
combined_point_cloud.colors.extend(flat_point_cloud.colors)
combined_point_cloud.colors.extend(half_ellipsoid_point_cloud.colors)


# --- Visualize the results ---
print("Visualizing mesh and generated point clouds (press 'q' to close the window)")
o3d.visualization.draw_geometries([mesh, combined_point_cloud])

# Optional: Visualize only the combined point cloud
# print("Visualizing combined point cloud (press 'q' to close the window)")
# o3d.visualization.draw_geometries([combined_point_cloud])

# You can now work with the 'combined_point_cloud' object in Open3D
# For example, you can access the points as a NumPy array:
# all_points = np.asarray(combined_point_cloud.points)
# all_colors = np.asarray(combined_point_cloud.colors)

Generated half-ellipsoid point cloud with 1500 points
Half-ellipsoid centered at [ 91.5    0.   584.67] with radii (1426.20, 664.80, 487.16)
Visualizing mesh and generated point clouds (press 'q' to close the window)


In [8]:
# --- Create the Spherical Point Cloud for the Cover ---
# Estimate the center and radius for the sphere.
# We'll center it above the car's center and give it a radius
# related to the car's dimensions.
sphere_center = center + np.array(
    [0, 0, (max_bound[2] - min_bound[2]) * 0.2]
)  # Shift center up by 20% of height
# Radius based on average of X and Y dimensions, with some padding
sphere_radius = (
    np.mean((max_bound - min_bound)[:2]) * 0.4 + (max_bound[2] - min_bound[2]) * 0.1
)  # 40% of avg XY + 10% of Z extent

# Number of points for the spherical cloud
n_sphere_points = 15000  # You can adjust this

# Generate random points within a cube that contains the sphere
cube_side_length = sphere_radius * 2.2  # Slightly larger than sphere diameter
cube_min_bound = sphere_center - cube_side_length / 2
cube_max_bound = sphere_center + cube_side_length / 2

random_points_in_cube = np.random.uniform(
    low=cube_min_bound, high=cube_max_bound, size=(n_sphere_points * 2, 3)
)  # Generate more points and filter

# Filter points to keep only those inside the sphere
distances = np.linalg.norm(random_points_in_cube - sphere_center, axis=1)
points_in_sphere = random_points_in_cube[distances < sphere_radius]
# If we didn't get enough points inside the sphere, generate more
if len(points_in_sphere) < n_sphere_points:
    print(
        f"Warning: Generated only {len(points_in_sphere)} points inside the sphere. Consider increasing initial random points."
    )
    # Use the points generated, or regenerate with a higher multiplier if needed

# Take the desired number of points if we generated more than needed
sphere_points = points_in_sphere[:n_sphere_points]


sphere_point_cloud = o3d.geometry.PointCloud()
sphere_point_cloud.points = o3d.utility.Vector3dVector(sphere_points)
sphere_point_cloud.paint_uniform_color([1, 0, 0])  # Paint spherical cloud red

print(
    f"Generated spherical point cloud with {len(sphere_point_cloud.points)} points around center {sphere_center.round(2)} with radius {sphere_radius:.2f}"
)

Generated spherical point cloud with 11735 points around center [ 91.5   0.  682.1] with radius 794.43


In [9]:
# --- Combine the Point Clouds ---
combined_point_cloud = o3d.geometry.PointCloud()
combined_point_cloud.points.extend(flat_point_cloud.points)
combined_point_cloud.points.extend(sphere_point_cloud.points)
combined_point_cloud.colors.extend(flat_point_cloud.colors)
combined_point_cloud.colors.extend(sphere_point_cloud.colors)


# --- Visualize the results ---
print("Visualizing mesh and generated point clouds (press 'q' to close the window)")
o3d.visualization.draw_geometries([mesh, combined_point_cloud])

# Optional: Visualize only the combined point cloud
# print("Visualizing combined point cloud (press 'q' to close the window)")
# o3d.visualization.draw_geometries([combined_point_cloud])

# You can now work with the 'combined_point_cloud' object in Open3D
# For example, you can access the points as a NumPy array:
# all_points = np.asarray(combined_point_cloud.points)
# all_colors = np.asarray(combined_point_cloud.colors)

Visualizing mesh and generated point clouds (press 'q' to close the window)
